# DROS Replication Experiment

**Paper:** A Robust Oversampling Approach for Class Imbalance Problem With Small Disjuncts  
**Authors:** Yi Sun, Lijun Cai, Bo Liao, Wen Zhu, Junlin Xu  
**Venue:** IEEE TKDE, 2023

Replicates DROS + SVM (Gaussian/RBF kernel) on all 17 UCI datasets from Table 1.  
Stratified 5-fold CV × 10 repeats = 50 runs per dataset.

In [1]:
import sys, os, time
import numpy as np
import pandas as pd

sys.path.insert(0, '.')
sys.path.insert(0, 'src')

from src import load_dataset, list_datasets, run_experiment
from src.data_loader import DATASET_REGISTRY

## 1. Available Datasets

In [2]:
list_datasets()

,name,description
0,haberman,Survival < 5yr
1,wpbc,Cancer wpbc ret
2,diabetes,Diabetes absent
3,hepatitis,Hepatitis normal
4,housing,Housing MEDV > 35
5,spectf,Spectf 0
6,iris,Iris setosa
7,abalone_5_6,Abalone_{5-6}
8,abalone_4_11,Abalone_{4-11}
9,ecoli_4_2,Ecoli_{4-2}


## 2. Run Experiment on All Datasets

DROS (ρ=0.5, δ=-0.7660, k=7, g=1) + SVM (Gaussian kernel, C=1, gamma='scale')  
Stratified 5-fold CV × 10 repeats = 50 runs per dataset

In [3]:
# Paper's reported DROS g-mean values (Table 2, SVM classifier)
PAPER_GMEAN = {
    'haberman': 0.5224, 'wpbc': 0.5750, 'diabetes': 0.7046,
    'hepatitis': 0.8099, 'housing': 0.8660, 'spectf': 0.7792,
    'iris': 1.0000, 'abalone_5_6': 0.6457, 'abalone_4_11': 0.7975,
    'ecoli_4_2': 0.9180, 'ecoli_5_1': 0.9757, 'glass_7_2': 0.9404,
    'glass_5_1': 0.7659, 'pageblocks_3_1': 0.9433, 'pageblocks_5_2': 0.8889,
    'yeast_5_3': 0.9512, 'yeast_9_4': 0.9685,
}

all_results = {}
all_summaries = {}
dataset_infos = {}

for name in DATASET_REGISTRY:
    print(f"Running {name}...", end=" ", flush=True)
    start = time.time()
    try:
        X, y, info = load_dataset(name)
        results, summary = run_experiment(X, y, verbose=False)
        elapsed = time.time() - start
        all_results[name] = results
        all_summaries[name] = summary
        dataset_infos[name] = info
        print(f"OK ({elapsed:.1f}s) — G-mean: {summary['g_mean']['mean']:.4f}")
    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nCompleted: {len(all_summaries)}/{len(DATASET_REGISTRY)} datasets")

Running haberman... OK (9.4s) — G-mean: 0.6193
Running wpbc... OK (4.7s) — G-mean: 0.1853
Running diabetes... OK (151.7s) — G-mean: 0.7311
Running hepatitis... OK (0.7s) — G-mean: 0.4220
Running housing... OK (5.8s) — G-mean: 0.6840
Running spectf... OK (6.6s) — G-mean: 0.6558
Running iris... OK (3.3s) — G-mean: 0.9949
Running abalone_5_6... OK (19.3s) — G-mean: 0.7172
Running abalone_4_11... OK (7.8s) — G-mean: 0.9843
Running ecoli_4_2... OK (2.0s) — G-mean: 0.7140
Running ecoli_5_1... OK (1.0s) — G-mean: 0.9727
Running glass_7_2... OK (1.6s) — G-mean: 0.9020
Running glass_5_1... OK (0.6s) — G-mean: 0.9871
Running pageblocks_3_1... OK (31.8s) — G-mean: 0.7616
Running pageblocks_5_2... OK (27.6s) — G-mean: 0.9331
Running yeast_5_3... OK (5.5s) — G-mean: 0.7972
Running yeast_9_4... OK (0.5s) — G-mean: 0.7200

Completed: 17/17 datasets


## 3. Results — Comparison With Paper (Table 2)

G-mean metric, DROS + SVM (Gaussian kernel).  
Paper uses MATLAB `fitcsvm`; we use sklearn `SVC(kernel='rbf', C=1, gamma='scale')`.

In [4]:
rows = []
for name, summary in all_summaries.items():
    info = dataset_infos[name]
    gm = summary['g_mean']['mean']
    gs = summary['g_mean']['std']
    paper = PAPER_GMEAN.get(name)
    diff = gm - paper if paper else None
    rows.append({
        'Dataset': info['description'],
        'Min:Maj': f"{info['n_minority']}:{info['n_majority']}",
        'IR': info['ir'],
        'Our G-mean': f"{gm:.4f} ± {gs:.4f}",
        'Paper G-mean': f"{paper:.4f}" if paper else "N/A",
        'Diff': f"{diff:+.4f}" if diff is not None else "N/A",
    })

comparison_df = pd.DataFrame(rows)
comparison_df

,Dataset,Min:Maj,IR,Our G-mean,Paper G-mean,Diff
0,Survival < 5yr,81:225,2.78,0.6193 ± 0.0693,0.5224,+0.0969
1,Cancer wpbc ret,46:148,3.22,0.1853 ± 0.1939,0.5750,-0.3897
2,Diabetes absent,268:500,1.87,0.7311 ± 0.0289,0.7046,+0.0265
3,Hepatitis normal,13:67,5.15,0.4220 ± 0.3339,0.8099,-0.3879
4,Housing MEDV > 35,48:458,9.54,0.6840 ± 0.1044,0.8660,-0.1820
5,Spectf 0,55:212,3.85,0.6558 ± 0.1011,0.7792,-0.1234
6,Iris setosa,50:100,2.00,0.9949 ± 0.0156,1.0000,-0.0051
7,Abalone_{5-6},115:259,2.25,0.7172 ± 0.0504,0.6457,+0.0715
8,Abalone_{4-11},57:487,8.54,0.9843 ± 0.0239,0.7975,+0.1868
9,Ecoli_{4-2},35:77,2.20,0.7140 ± 0.0835,0.9180,-0.2040


## 4. Full Metrics Table (All 5 Metrics)

In [5]:
rows = []
for name, summary in all_summaries.items():
    info = dataset_infos[name]
    row = {'Dataset': info['description']}
    for metric in ['precision', 'recall', 'f_measure', 'g_mean', 'auc']:
        m, s = summary[metric]['mean'], summary[metric]['std']
        row[metric.replace('_', ' ').title()] = f"{m:.4f} ± {s:.4f}"
    rows.append(row)

full_df = pd.DataFrame(rows)
full_df

,Dataset,Precision,Recall,F Measure,G Mean,Auc
0,Survival < 5yr,0.4453 ± 0.1017,0.5197 ± 0.1085,0.4720 ± 0.0848,0.6193 ± 0.0693,0.6939 ± 0.0666
1,Cancer wpbc ret,0.4083 ± 0.4512,0.0724 ± 0.0849,0.1190 ± 0.1345,0.1853 ± 0.1939,0.7089 ± 0.0964
2,Diabetes absent,0.5752 ± 0.0337,0.7768 ± 0.0601,0.6597 ± 0.0333,0.7311 ± 0.0289,0.8091 ± 0.0311
3,Hepatitis normal,0.5900 ± 0.4668,0.2933 ± 0.2726,0.3700 ± 0.2957,0.4220 ± 0.3339,0.8797 ± 0.1234
4,Housing MEDV > 35,0.9062 ± 0.1370,0.4813 ± 0.1446,0.6161 ± 0.1340,0.6840 ± 0.1044,0.9618 ± 0.0332
5,Spectf 0,0.6210 ± 0.1460,0.4800 ± 0.1470,0.5298 ± 0.1227,0.6558 ± 0.1011,0.8175 ± 0.0634
6,Iris setosa,1.0000 ± 0.0000,0.9900 ± 0.0303,0.9947 ± 0.0159,0.9949 ± 0.0156,1.0000 ± 0.0000
7,Abalone_{5-6},0.5504 ± 0.0761,0.7078 ± 0.0946,0.6141 ± 0.0602,0.7172 ± 0.0504,0.7773 ± 0.0575
8,Abalone_{4-11},0.9742 ± 0.0410,0.9726 ± 0.0469,0.9722 ± 0.0300,0.9843 ± 0.0239,0.9999 ± 0.0004
9,Ecoli_{4-2},0.5383 ± 0.1043,0.7571 ± 0.1392,0.6212 ± 0.0935,0.7140 ± 0.0835,0.7897 ± 0.0963


## 5. Per-Dataset Detail

Select a dataset to view per-fold statistics.

In [6]:
DETAIL_DATASET = 'glass_7_2'  # <-- Change this to any dataset name

if DETAIL_DATASET in all_results:
    detail = all_results[DETAIL_DATASET]
    print(f"Dataset: {dataset_infos[DETAIL_DATASET]['description']}")
    detail[['fold', 'precision', 'recall', 'f_measure', 'g_mean', 'auc']].describe()
else:
    print(f"'{DETAIL_DATASET}' not found in results. Available: {list(all_results.keys())}")

Dataset: Glass_{7-2}


## 6. Notes

- **Implementation:** DROS Algorithms 1–4 translated from paper pseudocode (Eqs. 8–25)
- **SVM:** sklearn `SVC(kernel='rbf', C=1.0, gamma='scale')` — closest match to MATLAB `fitcsvm` defaults after z-score standardization
- **Differences from paper:** Expected due to sklearn vs MATLAB SVM solver, random seed handling, and kernel scale heuristics. Largest gaps appear on very small datasets (Yeast_{9-4}: 5 minority samples) or extreme IR (Pageblocks_{3-1}: IR=175)